# Program 23
## Implement CRF for Named Entity Recognition
1. Setup sklearn-crfsuite
2. Train a CRF-based NER model
3. Feature extraction including prefix, suffix, word shape, and POS context.

In [ ]:
!pip install -q sklearn-crfsuite nltk

In [ ]:
import nltk
import sklearn_crfsuite
from sklearn_crfsuite import metrics

# Ensure required NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

print("Loading Data...")
# For demonstration purposes, we create a small CoNLL-like dataset structure.
# In a full environment, you might load CoNLL-2003 from `datasets` library like:
# from datasets import load_dataset; dataset = load_dataset('conll2003')
sample_sentences = [
    ["EU", "rejects", "German", "call", "to", "boycott", "British", "lamb", "."],
    ["Peter", "Blackburn", "is", "in", "London", "today", "."]
]

sample_ner_tags = [
    ["B-ORG", "O", "B-MISC", "O", "O", "O", "B-MISC", "O", "O"],
    ["B-PER", "I-PER", "O", "O", "B-LOC", "O", "O"]
]

train_sents = []
for words, labels in zip(sample_sentences, sample_ner_tags):
    pos_tags = [pos for word, pos in nltk.pos_tag(words)]
    train_sents.append(list(zip(words, pos_tags, labels)))

# Feature Extraction Function
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],      # Suffix
        'word[-2:]': word[-2:],
        'word[:3]': word[:3],        # Prefix
        'word[:2]': word[:2],
        'word.isupper()': word.isupper(), # Word Shape
        'word.istitle()': word.istitle(), # Word Shape
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
    }
    # Contextual features from previous word
    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True # Beginning of Sentence

    # Contextual features from next word
    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True # End of Sentence

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, postag, label in sent]

# Format Dataset
print("Extracting features...")
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

# Train CRF Model
print("Training CRF Model...")
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,    # Feature scaling / L1 regularization
    c2=0.1,    # Feature scaling / L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

# Evaluate
y_pred = crf.predict(X_train)

print("\n--- Model Performance (on training data for demonstration) ---")
labels = list(crf.classes_)
if 'O' in labels:
    labels.remove('O') # Ignore 'O' class in F1 calculation generally

metrics_report = metrics.flat_classification_report(
    y_train, y_pred, labels=labels, digits=3
)
print(metrics_report)

# Test on new sentence
test_words = ["Google", "operates", "in", "California", "."]
test_pos = [pos for word, pos in nltk.pos_tag(test_words)]
test_sent = list(zip(test_words, test_pos, ["O"]*len(test_words))) # Dummy labels

X_test = [sent2features(test_sent)]
y_pred_new = crf.predict(X_test)

print("\nTest Sentence Prediction:")
for word, pred in zip(test_words, y_pred_new[0]):
    print(f"{word}: {pred}")